In [1]:
import pandas as pd
import numpy as np
import re

print("Libraries imported successfully!")


Libraries imported successfully!


In [2]:
from google.colab import files

uploaded = files.upload()


Saving ted_talks_en.csv to ted_talks_en.csv


In [3]:
file_name = next(iter(uploaded))

df = pd.read_csv(file_name)

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)
df.head()


Dataset loaded successfully!
Dataset shape: (4005, 19)


,talk_id,title,speaker_1,all_speakers,occupations,about_speakers,views,recorded_date,published_date,event,native_lang,available_lang,comments,duration,topics,related_talks,url,description,transcript
0,1,Averting the climate crisis,Al Gore,{0: 'Al Gore'},{0: ['climate advocate']},{0: 'Nobel Laureate Al Gore focused the world’...,3523392,2006-02-25,2006-06-27,TED2006,en,"['ar', 'bg', 'cs', 'de', 'el', 'en', 'es', 'fa...",272.0,977,"['alternative energy', 'cars', 'climate change...","{243: 'New thinking on the climate crisis', 54...",https://www.ted.com/talks/al_gore_averting_the...,With the same humor and humanity he exuded in ...,"Thank you so much, Chris. And it's truly a gre..."
1,92,The best stats you've ever seen,Hans Rosling,{0: 'Hans Rosling'},{0: ['global health expert; data visionary']},"{0: 'In Hans Rosling’s hands, data sings. Glob...",14501685,2006-02-22,2006-06-27,TED2006,en,"['ar', 'az', 'bg', 'bn', 'bs', 'cs', 'da', 'de...",628.0,1190,"['Africa', 'Asia', 'Google', 'demo', 'economic...","{2056: ""Own your body's data"", 2296: 'A visual...",https://www.ted.com/talks/hans_rosling_the_bes...,You've never seen data presented like this. Wi...,"About 10 years ago, I took on the task to teac..."
2,7,Simplicity sells,David Pogue,{0: 'David Pogue'},{0: ['technology columnist']},{0: 'David Pogue is the personal technology co...,1920832,2006-02-24,2006-06-27,TED2006,en,"['ar', 'bg', 'de', 'el', 'en', 'es', 'fa', 'fr...",124.0,1286,"['computers', 'entertainment', 'interface desi...","{1725: '10 top time-saving tech tips', 2274: '...",https://www.ted.com/talks/david_pogue_simplici...,New York Times columnist David Pogue takes aim...,"(Music: ""The Sound of Silence,"" Simon & Garfun..."
3,53,Greening the ghetto,Majora Carter,{0: 'Majora Carter'},{0: ['activist for environmental justice']},{0: 'Majora Carter redefined the field of envi...,2664069,2006-02-26,2006-06-27,TED2006,en,"['ar', 'bg', 'bn', 'ca', 'cs', 'de', 'en', 'es...",219.0,1116,"['MacArthur grant', 'activism', 'business', 'c...",{1041: '3 stories of local eco-entrepreneurshi...,https://www.ted.com/talks/majora_carter_greeni...,"In an emotionally charged talk, MacArthur-winn...",If you're here today — and I'm very happy that...
4,66,Do schools kill creativity?,Sir Ken Robinson,{0: 'Sir Ken Robinson'},"{0: ['author', 'educator']}","{0: ""Creativity expert Sir Ken Robinson challe...",65051954,2006-02-25,2006-06-27,TED2006,en,"['af', 'ar', 'az', 'be', 'bg', 'bn', 'ca', 'cs...",4931.0,1164,"['children', 'creativity', 'culture', 'dance',...","{865: 'Bring on the learning revolution!', 173...",https://www.ted.com/talks/sir_ken_robinson_do_...,Sir Ken Robinson makes an entertaining and pro...,Good morning. How are you? (Audience) Good. It...


In [4]:
print("Column names:")
print(df.columns.tolist())

print("\nDataset information:")
df.info()


Column names:
['talk_id', 'title', 'speaker_1', 'all_speakers', 'occupations', 'about_speakers', 'views', 'recorded_date', 'published_date', 'event', 'native_lang', 'available_lang', 'comments', 'duration', 'topics', 'related_talks', 'url', 'description', 'transcript']

Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4005 entries, 0 to 4004
Data columns (total 19 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   talk_id         4005 non-null   int64  
 1   title           4005 non-null   object 
 2   speaker_1       4005 non-null   object 
 3   all_speakers    4001 non-null   object 
 4   occupations     3483 non-null   object 
 5   about_speakers  3502 non-null   object 
 6   views           4005 non-null   int64  
 7   recorded_date   4004 non-null   object 
 8   published_date  4005 non-null   object 
 9   event           4005 non-null   object 
 10  native_lang     4005 non-null   object 
 11  available_l

In [5]:
missing = df.isnull().sum()

missing = missing[missing > 0].sort_values(ascending=False)

print("Missing values by column:")
print(missing)

print("\nTotal missing cells:", df.isnull().sum().sum())


Missing values by column:
comments          655
occupations       522
about_speakers    503
all_speakers        4
recorded_date       1
dtype: int64

Total missing cells: 1685


In [6]:
df_prep = df.copy()

for col in ["occupations", "all_speakers"]:
    if col in df_prep.columns:
        df_prep[col] = df_prep[col].fillna("Unknown")

print("Missing text metadata handled.")


Missing text metadata handled.


In [7]:

content_cols = [
    "title",
    "topics",
    "description",
    "transcript",
    "about_speakers"
]

for col in content_cols:
    if col in df_prep.columns:
        df_prep[col] = df_prep[col].fillna("")

print("Missing content text handled.")

for col in content_cols:
    if col in df_prep.columns:
        print(col, "empty values:",
              df_prep[col].astype(str).str.strip().eq("").sum())


Missing content text handled.
title empty values: 0
topics empty values: 0
description empty values: 0
transcript empty values: 0
about_speakers empty values: 503


In [8]:

for col in ["views", "comments", "comments_numeric"]:
    if col in df_prep.columns:
        df_prep[col] = pd.to_numeric(
            df_prep[col], errors="coerce"
        )

print("Numerical columns converted where available.")

for col in ["views", "comments", "comments_numeric"]:
    if col in df_prep.columns:
        print(
            col,
            "| Missing:", df_prep[col].isna().sum(),
            "| Minimum:", df_prep[col].min(),
            "| Maximum:", df_prep[col].max()
        )


Numerical columns converted where available.
views | Missing: 0 | Minimum: 0 | Maximum: 65051954
comments | Missing: 655 | Minimum: 0.0 | Maximum: 6449.0


In [9]:

before = len(df_prep)

df_prep = df_prep.drop_duplicates()

after = len(df_prep)

print("Rows before:", before)
print("Exact duplicate rows removed:", before - after)
print("Rows after:", after)


Rows before: 4005
Exact duplicate rows removed: 0
Rows after: 4005


In [10]:
id_candidates = [
    col for col in df_prep.columns
    if col.lower() in ["talk_id", "id", "video_id"]
]

print("Possible ID columns:", id_candidates)

for col in id_candidates:
    print("\nColumn:", col)
    print("Missing IDs:", df_prep[col].isna().sum())
    print("Duplicate IDs:", df_prep[col].duplicated().sum())


Possible ID columns: ['talk_id']

Column: talk_id
Missing IDs: 0
Duplicate IDs: 0


In [11]:

text_cols = [
    "title",
    "description",
    "transcript",
    "about_speakers"
]

for col in text_cols:
    if col in df_prep.columns:
        df_prep[col] = (
            df_prep[col]
            .astype(str)
            .str.replace(r"\s+", " ", regex=True)
            .str.strip()
        )

print("Text formatting cleaned.")


Text formatting cleaned.


In [12]:
for col in ["recorded_date", "published_date"]:
    if col in df_prep.columns:
        df_prep[col] = pd.to_datetime(
            df_prep[col], errors="coerce"
        )

print("Date conversion completed.")

for col in ["recorded_date", "published_date"]:
    if col in df_prep.columns:
        print(
            col,
            "| Missing or invalid dates:",
            df_prep[col].isna().sum()
        )


Date conversion completed.
recorded_date | Missing or invalid dates: 1
published_date | Missing or invalid dates: 0


In [13]:

available_content_cols = [
    col for col in [
        "title", "topics", "description", "transcript"
    ]
    if col in df_prep.columns
]

if available_content_cols:
    df_prep["combined_text"] = (
        df_prep[available_content_cols]
        .fillna("")
        .astype(str)
        .agg(" ".join, axis=1)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

    print("Combined text created successfully!")
    print("Text columns used:", available_content_cols)
    print("Empty combined text rows:",
          df_prep["combined_text"].eq("").sum())
else:
    print("Expected content columns were not found.")


Combined text created successfully!
Text columns used: ['title', 'topics', 'description', 'transcript']
Empty combined text rows: 0


In [14]:

print("=" * 45)
print("FINAL PREPROCESSING REPORT")
print("=" * 45)

print("Final shape:", df_prep.shape)
print("Exact duplicate rows:", df_prep.duplicated().sum())

print("\nRemaining missing values:")
remaining_missing = df_prep.isnull().sum()
print(remaining_missing[remaining_missing > 0])

if "combined_text" in df_prep.columns:
    print(
        "\nEmpty combined text rows:",
        df_prep["combined_text"].eq("").sum()
    )

print("\nPreprocessing validation completed.")


FINAL PREPROCESSING REPORT
Final shape: (4005, 20)
Exact duplicate rows: 0

Remaining missing values:
recorded_date      1
comments         655
dtype: int64

Empty combined text rows: 0

Preprocessing validation completed.


In [15]:
df_prep["recorded_date"] = pd.to_datetime(
    df_prep["recorded_date"], errors="coerce"
)

df_prep["comments"] = pd.to_numeric(
    df_prep["comments"], errors="coerce"
)

comments_median = df_prep["comments"].median()

if pd.notna(comments_median):
    df_prep["comments"] = df_prep["comments"].fillna(
        comments_median
    )

print("Missing-value handling completed.")


Missing-value handling completed.


In [16]:

print("Remaining missing values:")
print(df_prep.isnull().sum()[df_prep.isnull().sum() > 0])

print("\nComments missing after handling:",
      df_prep["comments"].isna().sum())

print("Recorded date missing:",
      df_prep["recorded_date"].isna().sum())


Remaining missing values:
recorded_date    1
dtype: int64

Comments missing after handling: 0
Recorded date missing: 1


In [17]:
print("=" * 45)
print("FINAL PREPROCESSING REPORT")
print("=" * 45)

print("Final shape:", df_prep.shape)
print("Exact duplicate rows:", df_prep.duplicated().sum())

print("\nRemaining missing values:")
remaining_missing = df_prep.isnull().sum()
print(remaining_missing[remaining_missing > 0])

if "combined_text" in df_prep.columns:
    print(
        "\nEmpty combined text rows:",
        df_prep["combined_text"].eq("").sum()
    )

print("\nPreprocessing validation completed.")


FINAL PREPROCESSING REPORT
Final shape: (4005, 20)
Exact duplicate rows: 0

Remaining missing values:
recorded_date    1
dtype: int64

Empty combined text rows: 0

Preprocessing validation completed.


In [18]:
df_prep["recorded_date"] = pd.to_datetime(
    df_prep["recorded_date"],
    errors="coerce"
)

print("Missing recorded_date:",
      df_prep["recorded_date"].isna().sum())

print(df_prep.loc[
    df_prep["recorded_date"].isna(),
    ["title", "recorded_date"]
])


Missing recorded_date: 1
                                                  title recorded_date
3831  The unpaid work that GDP ignores -- and why it...           NaT


In [19]:
print("=" * 45)
print("FINAL PREPROCESSING REPORT")
print("=" * 45)

print("Final shape:", df_prep.shape)
print("Exact duplicate rows:", df_prep.duplicated().sum())

print("\nRemaining missing values:")
remaining_missing = df_prep.isnull().sum()
print(remaining_missing[remaining_missing > 0])

if "combined_text" in df_prep.columns:
    print(
        "\nEmpty combined text rows:",
        df_prep["combined_text"].eq("").sum()
    )

print("\nPreprocessing validation completed.")


FINAL PREPROCESSING REPORT
Final shape: (4005, 20)
Exact duplicate rows: 0

Remaining missing values:
recorded_date    1
dtype: int64

Empty combined text rows: 0

Preprocessing validation completed.
